# Personality Prediction: Introvert vs Extrovert

**Best Model:** Gradient Boosting
**Test Accuracy:** 0.9241 | **ROC-AUC:** 0.9578 | **Brier Score:** 0.0668

| Feature | Description |
|---|---|
| Time_spent_Alone | Hours/day alone (0-11) |
| Stage_fear | Yes/No |
| Social_event_attendance | Frequency 0-10 |
| Going_outside | Times/week 0-7 |
| Drained_after_socializing | Yes/No |
| Friends_circle_size | Count 0-15 |
| Post_frequency | Posts/week 0-10 |
| Personality | Target: Introvert / Extrovert |


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, cross_val_score,
                                      GridSearchCV, StratifiedKFold, learning_curve)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.calibration import calibration_curve
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, brier_score_loss,
                              ConfusionMatrixDisplay)
import shap

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
BLUE, GREEN, CORAL, AMBER, GRAY = '#185FA5','#3B6D11','#993C1D','#854F0B','#5F5E5A'

# FIX: column lists defined once here — used by both EDA and preprocessing
numeric_cols = ['Time_spent_Alone','Social_event_attendance','Going_outside',
                'Friends_circle_size','Post_frequency']
binary_cols  = ['Stage_fear','Drained_after_socializing']
all_feat     = numeric_cols + binary_cols   # matches ColumnTransformer output order

print("Setup complete.")

## 2. Load Data

In [ ]:
df = pd.read_csv('personality_dataset.csv')
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
# Data types and basic info
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_df = pd.DataFrame({'Count': missing, 'Pct%': (missing/len(df)*100).round(2)})
print("Missing values per column:")
print(missing_df)
print()
print("Imputation strategy:")
print("  Numeric columns -> median imputation (robust to outliers)")
print("  Binary columns  -> most_frequent imputation (Yes or No)")
print("  Both handled inside the sklearn Pipeline. No rows dropped.")

In [ ]:
# Target distribution
print("Target distribution:")
print(df['Personality'].value_counts())
print(f"\nBalance ratio: {df['Personality'].value_counts(normalize=True).round(3).to_dict()}")

In [ ]:
# All numeric features are discrete integer ordinal scales (Likert-style)
for col in numeric_cols:
    vals = sorted(df[col].dropna().unique().astype(int))
    print(f"{col:<35}: {vals}")

## 3. Exploratory Data Analysis

In [ ]:
pal = {'Introvert': BLUE, 'Extrovert': CORAL}

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle('Feature Distributions by Personality Type', fontsize=15, fontweight='bold')
for i, col in enumerate(numeric_cols):
    ax = axes[i//4][i%4]
    for lab, grp in df.groupby('Personality'):
        grp[col].dropna().hist(ax=ax, alpha=0.6, bins=12, label=lab, color=pal[lab])
    ax.set_title(col.replace('_',' '), fontweight='bold', fontsize=10)
    if i==0: ax.legend(fontsize=9)
for j, col in enumerate(binary_cols):
    ax = axes[1][j+1] if j==1 else axes[1][j]
    ct = df.groupby(['Personality', col]).size().unstack(fill_value=0)
    ct.T.plot(kind='bar', ax=ax, color=[BLUE,CORAL], edgecolor='white', legend=(j==0))
    ax.set_title(col.replace('_',' '), fontweight='bold', fontsize=10)
    ax.tick_params(axis='x', rotation=0)
ax = axes[1][3]
vc = df['Personality'].value_counts()
ax.pie(vc, labels=vc.index, autopct='%1.1f%%', colors=[BLUE,CORAL],
       startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
ax.set_title('Class Balance', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Box plots — shows separation between classes
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('Feature Box Plots by Personality Type', fontsize=13, fontweight='bold')
for i, col in enumerate(numeric_cols):
    df.boxplot(column=col, by='Personality', ax=axes[i],
               boxprops=dict(color=GRAY), medianprops=dict(color=CORAL, linewidth=2))
    axes[i].set_title(col.replace('_',' '), fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')
plt.suptitle('')
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap
df_enc = df.copy()
df_enc['Stage_fear']                = df_enc['Stage_fear'].map({'Yes':1,'No':0})
df_enc['Drained_after_socializing'] = df_enc['Drained_after_socializing'].map({'Yes':1,'No':0})
df_enc['Personality_num']           = (df_enc['Personality']=='Extrovert').astype(int)
df_num = df_enc.drop('Personality', axis=1).fillna(df_enc.median(numeric_only=True))

plt.figure(figsize=(10, 7))
corr = df_num.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1, annot_kws={'size': 9})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Preprocessing

In [ ]:
X = df.drop('Personality', axis=1)[all_feat]
y = (df['Personality'] == 'Extrovert').astype(int)

# All encoding inside pipeline — no training-serving skew
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
binary_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['No','Yes'],['No','Yes']],
                               handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('bin', binary_transformer,  binary_cols)
], remainder='drop')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Train extrovert ratio: {y_train.mean():.3f}")

In [ ]:
# FIX: fit only on X_train to verify null handling — no test data leakage
pre_check = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())]), numeric_cols),
    ('bin', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                      ('encoder', OrdinalEncoder(categories=[['No','Yes'],['No','Yes']],
                                                  handle_unknown='use_encoded_value', unknown_value=-1))]), binary_cols)
], remainder='drop')

X_train_check = pre_check.fit_transform(X_train)
X_test_check  = pre_check.transform(X_test)

print("Null check after pipeline transform (fit on train only):")
print(f"  Train NaN remaining : {np.isnan(X_train_check).any()}")
print(f"  Test  NaN remaining : {np.isnan(X_test_check).any()}")
print(f"  Train shape: {X_train_check.shape} — all {len(X_train)} rows retained")
print(f"  Test  shape: {X_test_check.shape}  — all {len(X_test)} rows retained")

## 5. Model Comparison — Default CV

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest':       RandomForestClassifier(random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=42),
    'SVM':                 SVC(probability=True, random_state=42)
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
default_results = {}
print(f"{'Model':<25} {'CV Acc':>8}  {'Std':>6}")
print("-"*45)
for name, model in models.items():
    pipe   = Pipeline([('pre', preprocessor), ('clf', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    default_results[name] = scores.mean()
    print(f"{name:<25} {scores.mean():.4f}  +/-{scores.std():.4f}")

## 6. Hyperparameter Tuning — All Models

In [ ]:
param_grids = {
    'Logistic Regression': {
        'model':  LogisticRegression(random_state=42, max_iter=1000),
        'params': {'clf__C': [0.01, 0.1, 1, 10, 100],
                   'clf__solver': ['lbfgs', 'liblinear']}
    },
    'Random Forest': {
        'model':  RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': {'clf__n_estimators':     [100, 200, 300],
                   'clf__max_depth':        [None, 10, 20],
                   'clf__min_samples_leaf': [1, 2, 5]}
    },
    'Gradient Boosting': {
        'model':  GradientBoostingClassifier(random_state=42),
        'params': {'clf__n_estimators':  [100, 200],
                   'clf__learning_rate': [0.05, 0.1, 0.2],
                   'clf__max_depth':     [3, 5]}
    },
    'SVM': {
        'model':  SVC(probability=True, random_state=42),
        'params': {'clf__C':      [0.1, 1, 10, 100],
                   'clf__gamma':  ['scale', 'auto'],
                   'clf__kernel': ['rbf', 'linear']}
    }
}

tuned_results  = {}
gs_objects     = {}   # store all gs objects for later inspection

print(f"{'Model':<22} {'Default CV':>10}  {'Tuned CV':>9}  {'Test Acc':>9}  {'ROC-AUC':>8}  {'Brier':>7}")
print("-"*75)
for name, cfg in param_grids.items():
    pipe = Pipeline([('pre', preprocessor), ('clf', cfg['model'])])
    gs   = GridSearchCV(pipe, cfg['params'], cv=cv, scoring='accuracy', n_jobs=-1)
    gs.fit(X_train, y_train)
    y_pred_i = gs.predict(X_test)
    y_prob_i = gs.predict_proba(X_test)[:,1]
    tuned_results[name] = {
        'tuned_cv':  gs.best_score_,
        'test_acc':  accuracy_score(y_test, y_pred_i),
        'roc_auc':   roc_auc_score(y_test, y_prob_i),
        'brier':     brier_score_loss(y_test, y_prob_i),
        'pipeline':  gs.best_estimator_,
        'params':    gs.best_params_
    }
    gs_objects[name] = gs
    print(f"{name:<22} {default_results[name]:.4f}      "
          f"{gs.best_score_:.4f}     {accuracy_score(y_test,y_pred_i):.4f}     "
          f"{roc_auc_score(y_test,y_prob_i):.4f}    {brier_score_loss(y_test,y_prob_i):.4f}")

In [ ]:
# Select best: accuracy first, ROC-AUC tiebreaker, Brier second tiebreaker
best_name = max(tuned_results, key=lambda k: (tuned_results[k]['test_acc'],
                                               tuned_results[k]['roc_auc'],
                                              -tuned_results[k]['brier']))
print(f"Best model  : {best_name}")
print(f"  Test Acc  : {tuned_results[best_name]['test_acc']:.4f}")
print(f"  ROC-AUC   : {tuned_results[best_name]['roc_auc']:.4f}")
print(f"  Brier     : {tuned_results[best_name]['brier']:.4f}")
print(f"  Params    : {tuned_results[best_name]['params']}")

best_model = tuned_results[best_name]['pipeline']
y_pred     = best_model.predict(X_test)
y_prob     = best_model.predict_proba(X_test)[:,1]
print()
print(classification_report(y_test, y_pred, target_names=['Introvert','Extrovert']))

In [ ]:
# FIX: top 5 combinations for the actual best model — not hardcoded to GBM
gs_best     = gs_objects[best_name]
gs_res_df   = pd.DataFrame(gs_best.cv_results_)
top5        = gs_res_df.sort_values('mean_test_score', ascending=False).head(5)
param_cols  = [c for c in top5.columns if c.startswith('param_')]
print(f"Top 5 parameter combinations for best model ({best_name}):")
print(top5[param_cols + ['mean_test_score','std_test_score']].to_string(index=False))

In [ ]:
# Default vs tuned bar + learning curve
names_list = list(tuned_results.keys())
fig, axes  = plt.subplots(1, 2, figsize=(18, 5))

x = np.arange(len(names_list)); w = 0.35
def_vals = [default_results[n]        for n in names_list]
tun_vals = [tuned_results[n]['tuned_cv'] for n in names_list]
clrs     = [BLUE, GREEN, CORAL, AMBER]

axes[0].bar(x-w/2, def_vals, w, label='Default CV',
            color=[c+'80' for c in clrs], edgecolor='white')
axes[0].bar(x+w/2, tun_vals, w, label='Tuned CV',
            color=clrs, edgecolor='white')
axes[0].set_ylim(0.88, 0.97); axes[0].set_xticks(x)
axes[0].set_xticklabels(names_list, rotation=12, fontsize=10)
axes[0].set_title('Default vs Tuned CV Accuracy', fontweight='bold')
axes[0].set_ylabel('Accuracy'); axes[0].legend()
for i, t in enumerate(tun_vals):
    axes[0].text(i+w/2, t+0.001, f'{t:.4f}', ha='center', fontsize=8, fontweight='bold')

cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_sz, tr_sc, val_sc = learning_curve(
    best_model, X, y, cv=cv_lc, scoring='accuracy',
    train_sizes=np.linspace(0.1,1.0,8), n_jobs=-1)
axes[1].plot(train_sz, tr_sc.mean(1),  'o-', color=CORAL, label='Train', lw=2)
axes[1].plot(train_sz, val_sc.mean(1), 'o-', color=BLUE,  label='Validation', lw=2)
axes[1].fill_between(train_sz, tr_sc.mean(1)-tr_sc.std(1),
                                tr_sc.mean(1)+tr_sc.std(1), alpha=0.1, color=CORAL)
axes[1].fill_between(train_sz, val_sc.mean(1)-val_sc.std(1),
                                val_sc.mean(1)+val_sc.std(1), alpha=0.1, color=BLUE)
axes[1].set_title(f'Learning Curve ({best_name})', fontweight='bold')
axes[1].set(xlabel='Training samples', ylabel='Accuracy')
axes[1].legend(); axes[1].set_ylim(0.85, 1.0)
plt.tight_layout(); plt.show()

## 7. Evaluation

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

# multi-metric comparison across all models
metrics = ['test_acc','roc_auc','tuned_cv']
mlabels = ['Test Accuracy','ROC-AUC','Tuned CV']
x = np.arange(len(names_list)); w2 = 0.25
for i,(m,ml) in enumerate(zip(metrics,mlabels)):
    vals = [tuned_results[n][m] for n in names_list]
    axes[0].bar(x+i*w2-w2, vals, w2, label=ml,
                color=[BLUE,GREEN,CORAL][i], alpha=0.85, edgecolor='white')
axes[0].set_ylim(0.88,1.0); axes[0].set_xticks(x)
axes[0].set_xticklabels(names_list, rotation=12, fontsize=9)
axes[0].set_title('All Models — Multiple Metrics', fontweight='bold')
axes[0].legend(fontsize=8)

# confusion matrix
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['Introvert','Extrovert']).plot(
    ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion Matrix', fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[2].plot(fpr, tpr, color=BLUE, lw=2.5, label=f'AUC={auc:.4f}')
axes[2].plot([0,1],[0,1],'k--',lw=1)
axes[2].fill_between(fpr,tpr,alpha=0.08,color=BLUE)
axes[2].set(xlabel='FPR',ylabel='TPR')
axes[2].set_title('ROC Curve',fontweight='bold'); axes[2].legend()

# calibration diagram — validates confidence scores returned by API
prob_true, prob_pred_cal = calibration_curve(y_test, y_prob, n_bins=8)
brier = brier_score_loss(y_test, y_prob)
axes[3].plot(prob_pred_cal, prob_true, 'o-', color=BLUE, lw=2, label=f'Brier={brier:.3f}')
axes[3].plot([0,1],[0,1],'k--',lw=1,label='Perfect')
axes[3].fill_between(prob_pred_cal,prob_true,prob_pred_cal,alpha=0.1,color=CORAL)
axes[3].set(xlabel='Mean predicted prob',ylabel='Fraction positives')
axes[3].set_title('Calibration Diagram',fontweight='bold'); axes[3].legend()
plt.tight_layout(); plt.show()

## 8. Feature Importance and SHAP

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Gini importance — correct column order (matches ColumnTransformer output)
fi      = best_model.named_steps['clf'].feature_importances_
feat_df = pd.DataFrame({'Feature':all_feat,'Importance':fi}).sort_values('Importance')
clrs_fi = [GREEN if v>0.15 else '#A8DADC' for v in feat_df['Importance']]
axes[0].barh(feat_df['Feature'].str.replace('_',' '), feat_df['Importance'], color=clrs_fi)
axes[0].set_title(f'Feature Importance ({best_name})', fontweight='bold')
axes[0].set_xlabel('Importance')
for i,v in enumerate(feat_df['Importance']):
    axes[0].text(v+0.003, i, f'{v:.3f}', va='center', fontsize=9)

# SHAP — accounts for feature interactions, more reliable than Gini
X_test_proc = best_model.named_steps['pre'].transform(X_test)
explainer   = shap.TreeExplainer(best_model.named_steps['clf'])
shap_vals   = explainer.shap_values(X_test_proc)
shap_mean   = np.abs(shap_vals).mean(0)
shap_df = pd.DataFrame({'Feature':all_feat,'SHAP':shap_mean}).sort_values('SHAP')
axes[1].barh(shap_df['Feature'].str.replace('_',' '), shap_df['SHAP'], color=AMBER)
axes[1].set_title('SHAP Mean |Value|', fontweight='bold')
axes[1].set_xlabel('Mean |SHAP value|')
for i,v in enumerate(shap_df['SHAP']):
    axes[1].text(v+0.001, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## 9. Error Analysis

In [ ]:
err_df = X_test.copy()
err_df['true']       = y_test.values
err_df['predicted']  = y_pred
err_df['confidence'] = np.max(best_model.predict_proba(X_test), axis=1)
err_df['correct']    = (err_df['true'] == err_df['predicted'])
errors  = err_df[~err_df['correct']]
correct = err_df[err_df['correct']]

print(f"Misclassified : {len(errors)} / {len(err_df)} ({len(errors)/len(err_df)*100:.1f}%)")
print(f"Mean confidence on errors : {errors['confidence'].mean():.3f}")
print(f"Mean confidence on correct: {correct['confidence'].mean():.3f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(correct['confidence'],bins=15,alpha=0.7,color=GREEN,label=f'Correct n={len(correct)}')
axes[0].hist(errors['confidence'], bins=15,alpha=0.7,color=CORAL,label=f'Errors n={len(errors)}')
axes[0].axvline(correct['confidence'].mean(),color=GREEN,linestyle='--',lw=2)
axes[0].axvline(errors['confidence'].mean(), color=CORAL,linestyle='--',lw=2)
axes[0].set(xlabel='Confidence',ylabel='Count')
axes[0].set_title('Confidence: Errors vs Correct',fontweight='bold'); axes[0].legend()

feat_compare = ['Time_spent_Alone','Social_event_attendance','Friends_circle_size']
x2 = np.arange(len(feat_compare)); w3=0.35
axes[1].bar(x2-w3/2,[errors[f].mean()  for f in feat_compare],w3,label='Errors', color=CORAL,alpha=0.8)
axes[1].bar(x2+w3/2,[correct[f].mean() for f in feat_compare],w3,label='Correct',color=GREEN,alpha=0.8)
axes[1].set_xticks(x2)
axes[1].set_xticklabels([f.replace('_','\n') for f in feat_compare],fontsize=9)
axes[1].set_title('Feature Means: Errors vs Correct',fontweight='bold'); axes[1].legend()

axes[2].hist(correct['Time_spent_Alone'],bins=12,alpha=0.6,color=GREEN,label='Correct')
axes[2].hist(errors['Time_spent_Alone'], bins=12,alpha=0.7,color=CORAL,label='Errors')
axes[2].axvline(5.5,color='black',linestyle='--',lw=1.5,label='Ambivert zone ~5-6')
axes[2].set(xlabel='Time spent alone',ylabel='Count')
axes[2].set_title('Errors cluster in ambivert zone',fontweight='bold'); axes[2].legend(fontsize=9)
plt.tight_layout(); plt.show()

## 10. Save Model

In [ ]:
joblib.dump(best_model, 'personality_model.pkl')
joblib.dump(all_feat,   'feature_names.pkl')

print(f"Model saved  : {best_name}")
print(f"Test Accuracy: {tuned_results[best_name]['test_acc']:.4f}")
print(f"ROC-AUC      : {tuned_results[best_name]['roc_auc']:.4f}")
print(f"Brier Score  : {tuned_results[best_name]['brier']:.4f}")
print(f"Best params  : {tuned_results[best_name]['params']}")

# Sanity check — raw Yes/No strings fed directly, no manual encoding needed
sample = pd.DataFrame([{
    'Time_spent_Alone': 8.0, 'Stage_fear': 'Yes',
    'Social_event_attendance': 1.0, 'Going_outside': 1.0,
    'Drained_after_socializing': 'Yes', 'Friends_circle_size': 2.0,
    'Post_frequency': 2.0
}])[all_feat]

pred  = best_model.predict(sample)[0]
proba = best_model.predict_proba(sample)[0]
label = 'Extrovert' if pred==1 else 'Introvert'
print(f"\nSample prediction : {label}")
print(f"P(Introvert)      : {proba[0]:.4f}")
print(f"P(Extrovert)      : {proba[1]:.4f}")

## 11. API Deployment

### Architecture
```
POST /predict
      |
      v
Pydantic validates schema and field ranges
      |
      v
Pipeline.predict_proba()   <- all encoding/imputation inside sklearn
      |
      v
Returns label + calibrated probabilities + model version
```



### Request
```bash
curl -X POST https://YOUR-URL.onrender.com/predict \
  -H "Content-Type: application/json" \
  -d '{
    "Time_spent_Alone": 7.0,
    "Stage_fear": "Yes",
    "Social_event_attendance": 2.0,
    "Going_outside": 2.0,
    "Drained_after_socializing": "Yes",
    "Friends_circle_size": 3.0,
    "Post_frequency": 2.0
  }'
```

### Response
```json
{
  "prediction": "Introvert",
  "confidence": 0.934,
  "probabilities": { "Introvert": 0.934, "Extrovert": 0.066 },
  "model_version": "1.0.0"
}
```
